In [1]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "jiwer", "pandas", "tqdm", "huggingface_hub"], check=True)

CompletedProcess(args=['c:\\Users\\ASUS\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', '-m', 'pip', 'install', '-U', 'jiwer', 'pandas', 'tqdm', 'huggingface_hub'], returncode=0)

In [10]:
original_model_path  = r"D:\code material 2.0\akshar-setu-v3-grav\models\indic-ocr"
finetuned_model_path = original_model_path        # same model for now     # later: r"D:\path\to\indic-ocr-degraded"

correct_dataset_path  = r"D:\code material 2.0\akshar-setu-v3-grav\modified_gt\eval_sets\modified"
modified_dataset_path = r"D:\code material 2.0\akshar-setu-v3-grav\eval_sets\modified"

N_LINES = None          # None = all lines; use 10 for a quick test
OUT_DIR = r"eval_results"

In [11]:
import os, sys, glob, re, json, gc, hashlib, time
import torch, jiwer, pandas as pd
from tqdm.auto import tqdm
from huggingface_hub import snapshot_download

def resolve(p):
    return p if os.path.isdir(p) else snapshot_download(p)

original_model_dir  = resolve(original_model_path)
finetuned_model_dir = resolve(finetuned_model_path)
for d in (original_model_dir, finetuned_model_dir):
    assert os.path.isdir(os.path.join(d, "weights", "ocr")), f"weights/ocr not found in {d}"

sys.path.insert(0, original_model_dir)
from indic_ocr import IndicOCR
os.makedirs(OUT_DIR, exist_ok=True)
print("baseline   model:", original_model_dir)
print("fine-tuned model:", finetuned_model_dir)

baseline   model: D:\code material 2.0\akshar-setu-v3-grav\models\indic-ocr
fine-tuned model: D:\code material 2.0\akshar-setu-v3-grav\models\indic-ocr


In [12]:
def load_dataset(folder, n=None):
    pngs = sorted(glob.glob(os.path.join(folder, "**", "*.png"), recursive=True))
    items = []
    for p in pngs:
        g = p[:-4] + ".gt.txt"
        if os.path.exists(g):
            items.append((p, open(g, encoding="utf-8-sig").read().strip()))
    return items[:n] if n else items

ds = {
    "Correct GT":  load_dataset(correct_dataset_path,  N_LINES),
    "Modified GT": load_dataset(modified_dataset_path, N_LINES),
}
for name, items in ds.items():
    print(f"{name:12s}: {len(items)} lines")

names_c = [os.path.basename(p) for p, _ in ds["Correct GT"]]
names_m = [os.path.basename(p) for p, _ in ds["Modified GT"]]
same_names = names_c == names_m
print("same image file names in both sets:", same_names)
n_diff = sum(a[1] != b[1] for a, b in zip(ds["Correct GT"], ds["Modified GT"])) if same_names else "n/a"
print("lines whose ground truth differs  :", n_diff)

Correct GT  : 200 lines
Modified GT : 200 lines
same image file names in both sets: True
lines whose ground truth differs  : 145


In [13]:
CACHE_FILE = os.path.join(OUT_DIR, "predictions_cache.json")
cache = json.load(open(CACHE_FILE, encoding="utf-8")) if os.path.exists(CACHE_FILE) else {}

def save_cache():
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False)

norm = lambda t: re.sub(r"\s+", " ", t.replace("▁", " ")).strip()

def md5(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

def fingerprint(model_dir):                       # changes if you overwrite the model weights
    files = glob.glob(os.path.join(model_dir, "weights", "ocr", "*"))
    return f"{sum(os.path.getsize(f) for f in files)}-{int(max(os.path.getmtime(f) for f in files))}"

def predict(model_dir, image_paths, desc):
    fp, parser, new = fingerprint(model_dir), None, 0
    hyps = []
    for p in tqdm(image_paths, desc=desc):
        key = f"{model_dir}|{fp}|{md5(p)}"
        if key not in cache:
            if parser is None:
                parser = IndicOCR.from_pretrained(model_dir)      # loaded only when needed
            cache[key] = norm(parser.parse(p)["markdown"])
            new += 1
            if new % 10 == 0:
                save_cache()
        hyps.append(cache[key])
    save_cache()
    if parser is not None:
        del parser
        gc.collect()
        torch.cuda.empty_cache()
    return hyps

In [6]:
def score(refs, hyps):
    wer, cer = jiwer.wer(refs, hyps), jiwer.cer(refs, hyps)
    return {
        "Word Accuracy": 1 - wer,
        "Character Accuracy": 1 - cer,
        "WER": wer,
        "CER": cer,
        "Exact-line match": sum(r == h for r, h in zip(refs, hyps)) / len(refs),
    }

In [14]:
MODELS = [("Baseline", original_model_dir), ("Fine-tuned", finetuned_model_dir)]
results, hyps_store = {}, {}
t0 = time.time()
for model_name, model_dir in MODELS:
    for ds_name, items in ds.items():
        refs = [g for _, g in items]
        hyps = predict(model_dir, [p for p, _ in items], desc=f"{model_name} | {ds_name}")
        hyps_store[(model_name, ds_name)] = hyps
        results[(model_name, ds_name)] = score(refs, hyps)
print(f"done in {(time.time() - t0) / 60:.1f} min")

Baseline | Correct GT: 100%|██████████| 200/200 [00:00<00:00, 3696.77it/s]
Baseline | Modified GT: 100%|██████████| 200/200 [00:00<00:00, 495.43it/s]
Fine-tuned | Correct GT: 100%|██████████| 200/200 [00:00<00:00, 5154.13it/s]
Fine-tuned | Modified GT: 100%|██████████| 200/200 [00:00<00:00, 5205.40it/s]

done in 0.0 min


In [15]:
model_names = [m for m, _ in MODELS]

cols = pd.MultiIndex.from_product([model_names, ["Word Acc", "Char Acc"]])
matrix = pd.DataFrame(index=list(ds), columns=cols, dtype=float)
for ds_name in ds:
    for m in model_names:
        r = results[(m, ds_name)]
        matrix.loc[ds_name, (m, "Word Acc")] = r["Word Accuracy"]
        matrix.loc[ds_name, (m, "Char Acc")] = r["Character Accuracy"]

metrics = ["Word Accuracy", "Character Accuracy", "WER", "CER", "Exact-line match"]
rows = []
for ds_name in ds:
    for met in metrics:
        b, f = results[("Baseline", ds_name)][met], results[("Fine-tuned", ds_name)][met]
        rows.append({"Dataset": ds_name, "Metric": met, "Baseline": b, "Fine-tuned": f, "Δ (FT − Base)": f - b})
detail = pd.DataFrame(rows).set_index(["Dataset", "Metric"])

print("ACCURACY MATRIX (%)")
print((matrix * 100).round(2).to_string())
print("\nDETAILED (%)")
print((detail * 100).round(2).to_string())

matrix.to_csv(os.path.join(OUT_DIR, "accuracy_matrix.csv"))
detail.to_csv(os.path.join(OUT_DIR, "detailed_metrics.csv"))
print("\nsaved to", os.path.abspath(OUT_DIR))

ACCURACY MATRIX (%)
            Baseline          Fine-tuned         
            Word Acc Char Acc   Word Acc Char Acc
Correct GT     80.41    96.38      80.41    96.38
Modified GT    72.34    94.88      72.34    94.88

DETAILED (%)
                                Baseline  Fine-tuned  Δ (FT − Base)
Dataset     Metric                                                 
Correct GT  Word Accuracy          80.41       80.41            0.0
            Character Accuracy     96.38       96.38            0.0
            WER                    19.59       19.59            0.0
            CER                     3.62        3.62            0.0
            Exact-line match       30.50       30.50            0.0
Modified GT Word Accuracy          72.34       72.34            0.0
            Character Accuracy     94.88       94.88            0.0
            WER                    27.66       27.66            0.0
            CER                     5.12        5.12            0.0
            Exact-

In [16]:
# CELL 8b — One comparison table: low score vs high score (same model)
from IPython.display import display

low  = results[("Baseline", "Modified GT")]   # scored against the altered labels
high = results[("Baseline", "Correct GT")]    # scored against the true labels

spec = [
    ("Word Accuracy (%)",      "Word Accuracy"),
    ("Character Accuracy (%)", "Character Accuracy"),
    ("WER (%)",                "WER"),
    ("CER (%)",                "CER"),
    ("Exact-line match (%)",   "Exact-line match"),
]

table = pd.DataFrame([
    {"Metric": label,
     "Baseline Model": round(low[key]  * 100, 2),
     "Finetuned Model":     round(high[key] * 100, 2),
     "Difference":                   round((high[key] - low[key]) * 100, 2)}
    for label, key in spec
]).set_index("Metric")

display(
    table.style
         .format({"Modified GT (altered labels)": "{:.2f}",
                  "Correct GT (true labels)": "{:.2f}",
                  "Difference": "{:+.2f}"})
         .set_caption("comaprision of baseline vs finetuned model")
)
table.to_csv(os.path.join(OUT_DIR, "comparison_table.csv"))

,Baseline Model,Finetuned Model,Difference
Metric,,,
Word Accuracy (%),72.340000,80.410000,+8.07
Character Accuracy (%),94.880000,96.380000,+1.49
WER (%),27.660000,19.590000,-8.07
CER (%),5.120000,3.620000,-1.49
Exact-line match (%),18.000000,30.500000,+12.50


In [29]:
if same_names:
    hyp = hyps_store[("Baseline", "Modified GT")]
    cor, mod = ds["Correct GT"], ds["Modified GT"]
    penalised = sum(h == c[1] and h != m[1] for h, c, m in zip(hyp, cor, mod))
    print(f"lines the model read correctly but the modified GT marks wrong: {penalised}")
    shown = 0
    for h, c, m in zip(hyp, cor, mod):
        if c[1] != m[1]:
            print("\nmodified GT:", m[1]); print("correct  GT:", c[1]); print("model      :", h)
            shown += 1
            if shown == 4: break

lines the model read correctly but the modified GT marks wrong: 162

modified GT: ભારું નામ રમેશ છે.
correct  GT: મારું નામ રમેશ છે.
model      : મારું નામ રમેશ છે.

modified GT: તમ કેભ છો?
correct  GT: તમે કેમ છો?
model      : તમે કેમ છો?

modified GT: આજે હવામાન ખબ સરસ છ.
correct  GT: આજે હવામાન ખૂબ સરસ છે.
model      : આજે હવામાન ખૂબ સરસ છે.

modified GT: મન પુસ્તકો વાંચવાનું ગમે છે.
correct  GT: મને પુસ્તકો વાંચવાનું ગમે છે.
model      : મને પુસ્તકો વાંચવાનું ગમે છે.
